# interpolated data from trimmed raw data 

In [32]:
import os
import glob
import numpy as np
import pandas as pd


#Save interpolated data csv

In [33]:
# import numpy as np

# def mark_long_gaps(gap_mask, target_vec, threshold):
#     gap_mask = np.asarray(gap_mask, dtype=bool)
#     target_vec = np.asarray(target_vec, dtype=float)  # allow NaN

#     # Find start and end of each gap run
#     diff = np.diff(gap_mask.astype(int))
#     starts = np.where(diff == 1)[0] + 1
#     ends   = np.where(diff == -1)[0] + 1

#     # Handle gap at beginning
#     if gap_mask[0]:
#         starts = np.r_[0, starts]

#     # Handle gap at end
#     if gap_mask[-1]:
#         ends = np.r_[ends, len(gap_mask)]

#     # Process runs
#     for s, e in zip(starts, ends):
#         if (e - s) >= threshold:
#             target_vec[s:e] = np.nan

#     return target_vec

In [34]:
# #Gap fill function

# def cubic_fill_gaps(data, max_gap=20):
#     # Force numeric conversion FIRST
#     data = pd.to_numeric(data, errors="coerce")

#     isnan = np.isnan(data)
#     N = len(data)
#     i = 0

#     while i < N:
#         if isnan[i]:
#             start = i
#             while i < N and isnan[i]:
#                 i += 1
#             end = i

#             gap = end - start
#             if gap <= max_gap and start > 1 and end < N - 1:
#                 x = np.array([start - 1, start, end - 1, end])
#                 y = data[x]

#                 # Extra safety: only interpolate if all anchor points exist
#                 if not np.any(np.isnan(y)):
#                     coeffs = np.polyfit(x, y, 3)
#                     poly = np.poly1d(coeffs)
#                     data[start:end] = poly(np.arange(start, end))
#         else:
#             i += 1

#     return data

In [35]:
# # add extra linear interpolation for head and wing bases

# def scipy_interpolate_1d(x, max_gap):
#     from scipy.interpolate import interp1d 
#     """
#     Linear interpolation using scipy.interpolate.interp1d
#     Interpolates NaN gaps <= max_gap
#     Does NOT interpolate at start or end
    
#     @2500 fps, and 250Hz wbf - a gap of 40 is about 4 wingbeats.
#     """
#     y = x.copy()
#     n = len(x)

#     isnan = np.isnan(x)
#     padded = np.r_[False, isnan, False]
#     diff = np.diff(padded.astype(int))

#     starts = np.where(diff == 1)[0]
#     ends   = np.where(diff == -1)[0]

#     for s, e in zip(starts, ends):
#         gap_len = e - s

#         if gap_len > max_gap:
#             continue

#         if s == 0 or e == n:
#             continue

#         idx = np.array([s - 1, e])
#         vals = y[idx]

#         if np.any(np.isnan(vals)):
#             continue

#         f = interp1d(idx, vals, kind='cubic')
#         y[s:e] = f(np.arange(s, e))

#     return y

## Find trimmed files

In [36]:
#Input 
INPUT_DIR  = r"./../../Tanvi-files/xyz_Raw_Trimmed"
OUTPUT_DIR = r"./../../Tanvi-files/xyz_Interpolated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_files = glob.glob(os.path.join(INPUT_DIR, "*_trimmed.csv"))
if not csv_files:
    raise FileNotFoundError("No trimmed CSVs found")

In [44]:
points_for_shorter_interpol = ['pt5', 'pt6']
short_column_list  = [cols for cols in df.columns if 
               any(k in cols for k in points_for_shorter_interpol)]
short_gap = [7] * len(short_column_list)

points_for_longer_interpol = ['pt1', 'pt2', 'pt3', 'pt4', 'center', 'pt7']
long_column_list = [cols for cols in df.columns if 
               any(k in cols for k in points_for_longer_interpol)]
long_gap = [20] * len(long_column_list)

columns = short_column_list + long_column_list
gaps = short_gap + long_gap
column_limits = dict(zip(columns, gaps))

In [45]:
column_limits

{'pt5_X': 7,
 'pt5_Y': 7,
 'pt5_Z': 7,
 'pt6_X': 7,
 'pt6_Y': 7,
 'pt6_Z': 7,
 'pt1_X': 20,
 'pt1_Y': 20,
 'pt1_Z': 20,
 'pt2_X': 20,
 'pt2_Y': 20,
 'pt2_Z': 20,
 'pt3_X': 20,
 'pt3_Y': 20,
 'pt3_Z': 20,
 'pt4_X': 20,
 'pt4_Y': 20,
 'pt4_Z': 20,
 'pt7_X': 20,
 'pt7_Y': 20,
 'pt7_Z': 20,
 'center_X': 20,
 'center_Y': 20,
 'center_Z': 20}

## run the code across all files

In [46]:
# Interpolate and save
for path in csv_files:
    fname = os.path.basename(path)
    print("Interpolating:", fname)

    df = pd.read_csv(path)
    
    # ---- interpolate all points - cubic ----
    interp_df = df.copy()
#     interp_df["frame"] = df.iloc[:, 0]
   
    

    # Apply the interpolation to each column with its specified limit
    for col, gaplength in column_limits.items():
        interp_df[col] = df[col].interpolate(method = 'cubicspline'
                                             ,limit = gaplength
                                            ,limit_area = 'inside')

# #     # interpolate wing tips across shorter chuncks
# #     points_for_shorter_interpol = ['pt5', 'pt6']
# #     column_list  = [cols for cols in df.columns if 
# #                    any(k in cols for k in points_for_shorter_interpol)]
# #     for col in column_list:
# #         interpol_func = interp1d(interp_df['frame'].values, df[col].values, kind = 'cubic')
# #         interp_df[col] = interpol_func[interp_df['frame']]
# #         interp_df[col] = mark_long_gaps(df[col], interp_df[col], 20)
# # #         interp_df[col] = scipy_interpolate_1d(df[col])

    
#     points_for_longer_interpol = ['pt1', 'pt2', 'pt3', 'pt4', 'center']
#     column_list = [cols for cols in df.columns if 
#                    any(k in cols for k in points_for_longer_interpol)]
#     for col in column_list:
#         interpol_func = interp1d(interp_df['frame'].values, df[col].values, kind = 'cubic')
#         interp_df[col] = interpol_func(interp_df['frame'].values)
#         interp_df[col] = mark_long_gaps(df[col], interp_df[col], 40)
        
    # --------------------------------------------------------------
    
    out_name = fname.replace("_trimmed.csv", "_INTERP.csv")
    interp_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

print("Interpolated CSVs saved.")

Interpolating: Trial2_200rpmxyzpts_trimmed.csv
Interpolating: Trial3_180rpmxyzpts_trimmed.csv
Interpolating: Trial4_400rpmxyzpts_trimmed.csv
Interpolating: Trial5_180rpmxyzpts_trimmed.csv
Interpolated CSVs saved.
